In [1]:
import time
import requests
from openai import OpenAI

In [2]:
BASE_URL = "http://vllm:8000/v1"
MODEL = "Qwen/Qwen3-VL-8B-Instruct"

# vLLM OpenAI-compatible mode accepts an arbitrary API key by default.
client = OpenAI(api_key="EMPTY", base_url=BASE_URL, timeout=600)

health = requests.get("http://vllm:8000/health", timeout=10)
print("Health endpoint status:", health.status_code)

Health endpoint status: 200


In [ ]:
import pandas as pd

sample = pd.read_csv("data.csv").sample(1)

#### Более точное определение цвета

In [3]:
image_url_1 = "https://img60.avito.ru/private/1/1.LkuborawgqKtC0Cny8JIddEDgKApB4SqLw.0qnesxFunxQuN_4sOcguYuLus1VowmYe3xaAkzdhORw?cqp=2.wZG7sIRwgcmEP_mf7h6Ef1HLVHndwPLdaZS0Q0NQPDhdfYRukeuiteA="
image_url_2 = "https://60.img.avito.st/image/1/1.p8lekbaw8abqMIkoKDX9hyrs9iDoMgsq6g.PEZe7VffnpFrSLdUUoja4xMmc_R6D_vaDTbT9GdVKGI"

strict_prompt = """
Ты оцениваешь 2 изображения товаров. Приоритет источников: изображения > текстовые поля.
Если изображения не были успешно загружены/прочитаны, верни строго:
{"error":"images_unavailable"}
Верни только JSON:
{
"vision_evidence": {
"товар_1": {"dominant_color": "...", "form": "...", "upholstery_guess": "..."},
"товар_2": {"dominant_color": "...", "form": "...", "upholstery_guess": "..."}
}
}
Критично:
- Если основной цвет товаров визуально различается, не называй их одинаковыми по цвету.
- Не подменяй визуальные признаки текстовыми, если они противоречат друг другу.
"""
messages_strict = [
{
  "role": "user",
  "content": [
      {"type": "image_url", "image_url": {"url": image_url_1}},
      {"type": "image_url", "image_url": {"url": image_url_2}},
      {"type": "text", "text": strict_prompt + "\n\n" + task_text},
  ],
}
]
start = time.time()
response_strict = client.chat.completions.create(
model=MODEL,
messages=messages_strict,
max_tokens=512,
temperature=0.0,
)
print(f"Latency strict: {time.time() - start:.2f}s")
print(response_strict.choices[0].message.content)

NameError: name 'task_text' is not defined

In [4]:
import json

def load_vis_params(vis_params_ans: str, parent_key: str) -> str:
    params = json.loads(vis_params_ans)["vision_evidence"][parent_key]
    return params['dominant_color'] + ", " + params['upholstery_guess'] + ", " + params['form']

In [5]:
prompt = """
Ты — строгий матчер товаров внутри категории (будет указана в начале задания). Оцени степень схожести двух товаров по шкале 1–5, где 5 — идентичные модели.  

СТРОГИЕ ПРАВИЛА ВЫВОДА - Верни ТОЛЬКО валидный JSON-объект БЕЗ текста до/после и без лишних полей:   
{     
    "reasoning": "<кратко, по пунктам в одну строку; какие проверки прошли/не прошли; что извлечено из null>",
    "Класс": 1|2|3|4|5
} 
- "reasoning" обязателен и непустой. "Класс" — целое число 1..5. Никаких бэктиков, пояснений, метаданных.  

Сущности которые будут тебе доступны: 
- список ключевых параметров для категории 
- поле "заголовок и описание" товара 1
- поле "заголовок и описание" товара 2
- возможные значения ключевых параметров для товара 1 в поле "Возможные параметры товара 1" 
- возможные значения ключевых параметров для товара 2 в поле "Возможные параметры товара 2" 
- визуально распознанные значения ключевых атрибутов товара 1 в поле "Атрибуты по фото товара 1"
- визуально распознанные значения ключевых атрибутов товара 2 в поле "Атрибуты по фото товара 2"

ЛОГИКА РЕШЕНИЯ: 
- по каждому ключевому параметру из поля "заголовок и описание" товара 1 извлеки их значения и запомни их как "параметр: llm значение". 
Если из названия нельзя найти значение не запоминай такой параметр. 
- определи значения каждого ключевого параметра для товара 1 и товара 2 исходя из вида на изображении и запомни их как "параметр: image значение". 
- по каждому ключевому параметру слей "llm значение" с "возможными значения ключевых параметров" из полей "Возможные параметры товара 1".
Запомни их как "параметр: точное значение" для товара 1.
- по каждому ключевому параметру слей "llm значение" с "возможными значения ключевых параметров" из полей "Возможные параметры товара 2" если значение в "Возможные параметры товара 2" пустое, иначе используй "Возможные параметры товара 2".
Запомни их как "параметр: точное значение" для товара 2.
- определи значения каждого ключевого параметра из поля "Атрибуты по фото товара 1" и "Атрибуты по фото товара 2".
Запомни их как "визуальный_параметр: точное значение"
В итоге у тебя будет 4 словаря (добавь их в reasoning в ответе):     
    - "параметр: точное значение" для товара 1     
    - "параметр: точное значение" для товара 2
    - "визуальный_параметр: точное значение" для товара 1
    - "визуальный_параметр: точное значение" для товара 2

- начни отвечать на ЛОГИКУ КАСКАДА по вопросам, чтобы определить класс:     
    ЛОГИКА КАСКАДА (двигайся сверху вниз; на первом критичном «нет» — верни предыдущий класс)     
    Класс 1 — Базовая сопоставимость:     
    В1) Оба объекта — товары     
    - Нет → Класс 1; Да → В2.     
    Класс 2 — Категория:     
    В2) Оба товара относятся к одной категории? (Категория указана в начале задания)     
    - Нет → Класс 2; Да → В3.     
    Класс 3 — Сильное сходство параметров:    
    В3) Ключевые параметры для категории согласованы и близки после нормализации?     
    Список основных параметров будет в начале задания. Параметры могут отсутствовать, в этом случае стоит перейти к следующему вопросу. 
    Мы можем проигнорировать конфликт в значениях параметра "форма", если они близки по геометрии (например, угловой ≈ п-образный).
    - Нет (мало совпадений/есть конфликты) → Класс 2; Да → В4.
    Класс 4 — Сильное сходство по фото:
    В4) Атрибуты по фото совпадают?
    Цвета по фото точно совпадают, обивка по фото , формы по фото близки (например, угловой ≈ п-образный)?
    - Нет → Класс 3; Да → В5.
    В4) Изображения товаров сильно похожи, ключевые параметры тоже совпадают после нормализации?     
    - Да → Класс 5; Иначе (есть отличия по ключевым параметрам, признаку) → Класс 4.  
    
НОРМАЛИЗАЦИЯ 
- Модельные коды: игнорируй регистр/пробелы/дефисы/подчёркивания/точки: "ABC-123" == "abc 123" == "ABC_123". 
- Транслит в кодах: А↔️A, В↔️B, Е↔️E, К↔️K, М↔️M, Н↔️H, О↔️O, Р↔️P, С↔️C, Т↔️T, У↔️Y, Х↔️X; 0↔️O — только в модельном контексте. 
- Единицы/числа: ГБ~GB, ТБ~TB, МБ~MB; см/мм/дюйм; В~Volt; Вт~W; мА·ч~mAh — приводи к числам и сравнивай по значению. 
- Допустимая погрешность ≤1% (кроме дискретных модельных параметров — 0%; напр. память 128 vs 256 ГБ). 
- Игнорируй неключевые различия: цвет, регион упаковки (EU/CN/US), подарки/бандлы, продавец, маркетинговые слова.  

ОТВЕТ 
- Верни ровно один JSON-объект с полями "reasoning" и "Класс" (1..5). Ничего больше не выводи.

"""

vis_params_1 = load_vis_params(response_strict.choices[0].message.content, "товар_1")
vis_params_2 = load_vis_params(response_strict.choices[0].message.content, "товар_2")

task_text = f"""
\n\n
Текст задания:
Категория товаров: Диваны 
Список ключевых параметров для категории: материал обивки, форма, основной цвет 

Товар 1: 
заголовок: диван зеленый угловой В
Возможные параметры товара 1: велюр, угловой, зелёный
Атрибуты по фото товара 1: {vis_params_1}

Товар 2: 
заголовок: dreamery design модульный диван, механизм еврокнижка, 310х140х43 см,зеленый 
Возможные параметры товара 2: велюр, п-образный, зелёный
Атрибуты по фото товара 2: {vis_params_2}
"""

messages = [
    {
        "role": "user",
        "content": [
            {"type": "image_url", "image_url": {"url": image_url_1}},
            {"type": "image_url", "image_url": {"url": image_url_2}},
            {
                "type": "text",
                "text": prompt + task_text
            },
        ],
    }
]

NameError: name 'response_strict' is not defined

In [61]:
start = time.time()
response = client.chat.completions.create(
    model=MODEL,
    messages=messages,
    max_tokens=512,
    temperature=0.1,
    top_p=0.3
)
elapsed = time.time() - start

print(f"Latency: {elapsed:.2f}s")
print("\nModel response:\n")
print(response.choices[0].message.content)

Latency: 2.88s

Model response:

{
    "reasoning": "Параметр 'материал обивки': товар 1 — велюр (из возможных и визуальных), товар 2 — велюр (из возможных и визуальных). Параметр 'форма': товар 1 — п-образный (визуально), товар 2 — п-образный (визуально). Параметр 'основной цвет': товар 1 — бежевый (визуально), товар 2 — зеленый (визуально). Ключевые параметры 'материал обивки' и 'форма' совпадают, 'основной цвет' отличается. Согласно логике каскада: класс 3 — сильное сходство параметров, так как 'материал обивки' и 'форма' совпадают, а 'основной цвет' — отличается, но не влияет на класс 3, так как в логике каскада для класса 3 требуется только сильное сходство параметров, а не полное совпадение. В4 не выполняется, так как цвета по фото отличаются (бежевый vs зеленый).",
    "Класс": 3
}
